In [ ]:
import json
import os
import pandas as pd

# Initialize lists to store extracted coordinates
marker1_x, marker1_y = [], []  # Right Hip
marker2_x, marker2_y = [], []  # Right Knee
marker3_x, marker3_y = [], []  # Right Ankle

# Define joint indexes based on OpenPose's output
hip_idx = 9
knee_idx = 10
ankle_idx = 11

# Path to the folder containing OpenPose JSON files
json_folder_path = 'D:\Tugas Akhir\Subject TA\S49\OpenPose'

# Check if the folder exists and contains JSON files
if os.path.exists(json_folder_path):
    json_files = [f for f in os.listdir(json_folder_path) if f.endswith('.json')]
    
    if not json_files:
        print("No JSON files found in the specified folder.")
    else:
        # Sort the files to ensure they are processed in order
        for file_name in sorted(json_files):
            file_path = os.path.join(json_folder_path, file_name)
            
            print(f"Processing file: {file_name}")

            # Read the JSON file
            with open(file_path, 'r') as json_file:
                json_content = json.load(json_file)

                # Check if people are detected in the frame
                if not json_content.get('people'):
                    print("No persons detected in this frame.")
                    marker1_x.append(None)
                    marker1_y.append(None)
                    marker2_x.append(None)
                    marker2_y.append(None)
                    marker3_x.append(None)
                    marker3_y.append(None)
                    continue
                
                # Find the individual with the highest confidence score
                max_confidence = -1
                keypoints_for_max_confidence = None
                
                for person in json_content['people']:
                    keypoints = person['pose_keypoints_2d']
                    avg_confidence = sum(keypoints[i * 3 + 2] for i in [hip_idx, knee_idx, ankle_idx]) / 3
                    
                    if avg_confidence > max_confidence:
                        max_confidence = avg_confidence
                        keypoints_for_max_confidence = keypoints
                
                # Extract the coordinates for the right leg joints
                if keypoints_for_max_confidence:
                    marker1_x.append(keypoints_for_max_confidence[hip_idx * 3])
                    marker1_y.append(keypoints_for_max_confidence[hip_idx * 3 + 1])
                    marker2_x.append(keypoints_for_max_confidence[knee_idx * 3])
                    marker2_y.append(keypoints_for_max_confidence[knee_idx * 3 + 1])
                    marker3_x.append(keypoints_for_max_confidence[ankle_idx * 3])
                    marker3_y.append(keypoints_for_max_confidence[ankle_idx * 3 + 1])
                else:
                    marker1_x.append(None)
                    marker1_y.append(None)
                    marker2_x.append(None)
                    marker2_y.append(None)
                    marker3_x.append(None)
                    marker3_y.append(None)
else:
    print("The specified folder does not exist.")

# Create a DataFrame and save it to a CSV file
if marker1_x:
    df = pd.DataFrame({
        'Marker1x': marker1_x,
        'Marker1y': marker1_y,
        'Marker2x': marker2_x,
        'Marker2y': marker2_y,
        'Marker3x': marker3_x,
        'Marker3y': marker3_y,
    })

    csv_file_path = 'D:\Tugas Akhir\Subject TA\S49\OpenPose csv\S49_OPENPOSE.csv'
    df.to_csv(csv_file_path, index=False)
    print(f"Data saved to {csv_file_path}")
else:
    print("No data to save.")
